<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/Naive_Bayes_word_with_char.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# NAIVE BAYES - (WORD + CHAR COMBINATION)
# ==========================================================

import pandas as pd
import re
import string
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from scipy.sparse import hstack

# ==========================================================
# 1. LOAD DATA SPLITS
# ==========================================================
train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ==========================================================
# 2. STOPWORDS
# ==========================================================
afan_oromo_stopwords = {
    "fi","kan","akka",'akka','waan','isa','kan','keessa','irra','itti',
    'aanee','gidduu','narraa','gubbaa','ittuu','natti','akkam','hanga','hamma','jala',
    'nu','nurraa','jara','hennaa','akkasumas','akkuma','hogguu','sana','nuti','ala',
    'illee','siin','alatti','immoo','kana','silaa','sila','amma','inni','kanaafi',
    'kanaaf','sitti','ammo','kanaafuu','sun','an','ani','isaaf','keenna','tanaafuu',
    'ati','isaanirraa','dha','bira','isatti','keessan','teenya','booda','tun','ishiirraa',
    'kun','yommuu','keenya','utuu','duuba','koo','yeroo','moo','eega','eegasii',
    'isii','malee','booddee','dura','ishiif','yoo','fi','isin','na','yookaan',
    'gama','isiin','isheen','ishii','naaf','isaanif','anaaf','erge','isheerraa','isinirraa',
    'garuu','yookiin','keessatti','warra','yoom','anarraa','sirraa','ykn','fakkeenyaaf',"ta'a",
    'kiyya','kanamalees','jidduu',"haa", "ture", "turte",  "jira", "jiru", "jirtu", "jedha",
    "jedhe", "jedhan"
}

safe_stopwords = afan_oromo_stopwords - {"hin", "miti"}

# ==========================================================
# 3. TEXT PREPROCESSING
# ==========================================================
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()                             # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)           # Remove URLs
    text = re.sub(r'@\w+', '@user', text)                # Normalize user mentions
    text = re.sub(r'[\U0001F600-\U0001FFFF]', '', text)  # Remove all emojis
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)           # Collapse repeated letters  to two(gaariii → gaarii)
    text = re.sub(r"[^\w\s']", '', text)                 # Remove punctuation and keep apostrophe
    text = ' '.join(text.split())                        # Normalize spaces

    return text

def remove_stopwords(text):
    return " ".join([w for w in text.split() if w not in safe_stopwords])

# Apply preprocessing
for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["Text"].apply(clean_text)
    df["clean_text"] = df["clean_text"].apply(remove_stopwords)

# ==========================================================
# 4. TF-IDF FEATURE EXTRACTION (WORD + CHAR COMBINED)
# ==========================================================

# WORD TF-IDF
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

# CHARACTER TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=10000,
    min_df=2
)

# Fit on training data
X_train_word = word_vectorizer.fit_transform(train_df["clean_text"])
X_val_word   = word_vectorizer.transform(val_df["clean_text"])
X_test_word  = word_vectorizer.transform(test_df["clean_text"])

X_train_char = char_vectorizer.fit_transform(train_df["clean_text"])
X_val_char   = char_vectorizer.transform(val_df["clean_text"])
X_test_char  = char_vectorizer.transform(test_df["clean_text"])

# Combine features
X_train = hstack([X_train_word, X_train_char])
X_val   = hstack([X_val_word, X_val_char])
X_test  = hstack([X_test_word, X_test_char])

y_train = train_df["label"]
y_val   = val_df["label"]
y_test  = test_df["label"]

# ==========================================================
# 5. TRAIN MODEL
# ==========================================================
model = MultinomialNB()
model.fit(X_train, y_train)

# ==========================================================
# 6. PREDICTION
# ==========================================================
preds = model.predict(X_test)

# ==========================================================
# 7. EVALUATION
# ==========================================================
accuracy = accuracy_score(y_test, preds)

print("\n===============================")
print("NAIVE BAYES (WORD + CHAR)")
print("===============================")
print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:\n")
print(classification_report(y_test, preds, target_names=["Negative", "Neutral", "Positive"]))

cm = confusion_matrix(y_test, preds)

print("\nConfusion Matrix:\n", cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(cmap="Blues")

plt.title("Naïve Bayes Confusion Matrix (Word + Char)")
plt.show()